# Melify
At the moment, because it is rather complecated, we shall first take the test set and get the mean 
and variance. In this way we can proceed without conducting any preprocessing. The only change is that we will fix the normalization here and pass in the data for all trainings and testings. 

However, because at the moment I haven't found a good way to preserve good variance if using online methods, I will use the mean and variance as a whole. Here we take a stat of the mean and variance. 

In [1]:
from model_dataset import SaShiDatasetManualNorm
from model_dataset import MelSpecTransformDBNoNorm as TheTransform
from paths import *
from model_dataset import Normalizer, DeNormalizer, TokenMap

import pandas as pd
import pickle

In [2]:
transform_configs = {
    "sample_rate": 16000,
    "n_fft": 512,
    "hop_length": 128,
    "n_mels": 96,  
}

model_configs = {
    "input_dim": 96,   # this must equal to n_mels
    "output_dim": 96, 
    "inter_dim_0": 512,
    "dropout": 0.5, 
    "num_layers": 5,
}

train_configs = {
    "batch_size": 128,
    "num_epochs": 100,
    "num_workers": 32,
    "learning_rate": 5e-4,
}

In [3]:
t_set = pd.read_csv(os.path.join(src_, "phi-sashi-S-guide.csv"))
st_set = pd.read_csv(os.path.join(src_, "phi-sashi-Sh-guide.csv"))

t_set_sampled = t_set.sample(n=len(st_set))

integrated = pd.concat([t_set_sampled, st_set], ignore_index=True, sort=False)
integrated = integrated.sample(frac=1).reset_index(drop=True)

In [4]:
mytrans = TheTransform(sample_rate=transform_configs["sample_rate"], hop_length=transform_configs["hop_length"],
                       n_mels=transform_configs["n_mels"], n_fft=transform_configs["n_fft"])

with open(os.path.join(src_, "no-stress-seg.dict"), "rb") as file:
    # Load the object from the file
    mylist = pickle.load(file)
    mylist = ["BLANK"] + mylist
    mylist = mylist + ["SIL"]

# Now you can use the loaded object
mymap = TokenMap(mylist)

mynorm = Normalizer(Normalizer.norm_mvn_manual)

In [5]:
ds = SaShiDatasetManualNorm(
    src_dir=train_cut_phone_, guide_=integrated, 
    mapper=mymap, transform=mytrans, normalizer=mynorm, 
    noise_fixlength=False, noise_amplitude_scale=0.004, mv_config=None
)

No mean and variance provided, calculating from the data ...


In [6]:
ds.mean.item(), ds.std.item()

(-14.269546508789062, 15.378486633300781)

In [ ]:
mean_std_dict = {'mean': ds.mean.item(), 'std': ds.std.item()}

with open(os.path.join(src_, "mv_config_sashi_512_128_96.pkl"), "wb") as file:
    pickle.dump(mean_std_dict, file)

: 

Now we have saved the mean and variance, we can use them to normalize the data. 